# Solving a QUBO problem easily

Solving a QUBO problem is straightforward with `qubo-solver`: build an `Instance`, then call a classical solver, or run the quantum pipeline (embedding, drive shaping, compiling and running on a backend). Each step is a plain function with sensible defaults.
We have more advanced tutorials on the quantum-related components to dive deeper into these concepts.

## Solving with a quantum approach

To use a quantum approach, several choices have to be made, explained in more details in the [Solving QUBOs section of the documentation](https://pasqal-io.github.io/qubo-solver/latest/content/solving/).

One main decision is about the [backend](https://pasqal-io.github.io/qubo-solver/latest/content/backend/), that is how we choose to perform quantum runs. We can decide to either perform our on emulators (locally, or remotely) or using a real quantum processing unit (QPU). Our QPU, based on the Rydberg Analog Model, is accessible remotely.

### Available backend types and devices

The supported backends are available via [`qoolqit`](https://pasqal-io.github.io/qoolqit/latest/api/qoolqit/execution/backends/), a Python package designed for algorithm development in the Rydberg Analog Model.

The backends can be divided into 3 main categories:

- [Local emulators](https://pasqal-io.github.io/emulators/latest/)
- [Remote emulators](https://docs.pasqal.com/cloud/emu-tn/), which can be accessed via [`pasqal_cloud`](https://docs.pasqal.com/cloud/)
- [A remote QPU, such as Fresnel](https://docs.pasqal.com/cloud/fresnel-job/)

A backend will use device specifications to perform quantum computations. The list of supported devices can be found in [the QoolQit devices documentation](https://pasqal-io.github.io/qoolqit/latest/api/qoolqit/devices/).

### Running locally with an emulator

We can perform quantum simulations locally via an emulator.

In [ ]:
from __future__ import annotations

import numpy as np
import qoolqit

from qubosolver import (
    Instance,
    LocalEmulator,
    Solution,
    analysis,
    drive_shaping,
    embedding,
    solving,
)

Q = np.asarray(
    [
        [-0.2, 0.0, 1.0],
        [0.0, -1.0, 1.5],
        [1.0, 1.5, -0.1],
    ],
    dtype=np.float64,
)
instance = Instance(Q)

# Run the quantum pipeline on a local emulator.
device = qoolqit.AnalogDeviceWithDMM()
backend = LocalEmulator()

register = embedding.blade.embed_for_device(instance, device)
drive = drive_shaping.local_energy_scale.build_drive(instance, register, device=device)
program = solving.analog_quantum_sampling.compile(register, drive, device)
job = backend.run(program)
solution = Solution.from_results(job.results(), instance)

print(analysis.to_dataframe([solution]))

  labels bitstrings  costs  counts  probs
0      0        110   -1.2     729  0.729
1      0        010   -1.0     210  0.210
2      0        100   -0.2      45  0.045
3      0        001   -0.1      13  0.013
4      0        000    0.0       3  0.003


### Running with a remote connection

We can decide to perform our runs remotely via [`pasqal_cloud`](https://docs.pasqal.com/cloud/).
To do so, we have to provide several information after [setting up an account](https://docs.pasqal.com/cloud/set-up/).

#### On a real QPU

The code above can be modified to solve the QUBO instance using our real QPU remotely as follows (run only with your `pasqal_cloud` information):

In [ ]:
import numpy as np
import qoolqit
from pasqal_cloud import PasqalCloudConnection
from qoolqit.execution import QPU

from qubosolver import (
    Instance,
    Solution,
    analysis,
    drive_shaping,
    embedding,
    solving,
)

# Replace with your username, project id and password on the Pasqal Cloud.
USERNAME = "#TO_PROVIDE"
PROJECT_ID = "#TO_PROVIDE"
PASSWORD = None

Q = np.asarray(
    [
        [-0.2, 0.0, 1.0],
        [0.0, -1.0, 1.5],
        [1.0, 1.5, -0.1],
    ],
    dtype=np.float64,
)
instance = Instance(Q)

if PASSWORD is not None:
    # Setup connection
    connection = PasqalCloudConnection(
        username=USERNAME,
        password=PASSWORD,
        project_id=PROJECT_ID,
    )
    # Get available devices
    print(f"Available devices: {connection.fetch_available_devices()}")
    # Choose a device, and use a quantum backend
    device = qoolqit.Device.from_connection(connection, "FRESNEL_CAN1")
    qpu_backend = QPU(connection=connection, num_shots=1000)
else:
    # Use a mock local connection for the tutorial
    from qubosolver import RemoteEmulator
    from qubosolver.utils._local_connection import LocalConnection

    connection = LocalConnection()
    device = qoolqit.AnalogDevice()
    qpu_backend = RemoteEmulator(connection=connection)

backend = qpu_backend

register = embedding.blade.embed_for_device(instance, device)
drive = drive_shaping.local_energy_scale.build_drive(instance, register, device=device)
program = solving.analog_quantum_sampling.compile(register, drive, device)
job = backend.run(program)
solution = Solution.from_results(job.results(), instance)

print(analysis.to_dataframe([solution]))

  labels bitstrings  costs  counts  probs
0      0        110   -1.2     893  0.893
1      0        010   -1.0      15  0.015
2      0        100   -0.2      20  0.020
3      0        001   -0.1      71  0.071
4      0        000    0.0       1  0.001


#### On a remote emulators

Emulators are also available remotely via `pasqal_cloud`:

In [ ]:
import numpy as np
from pasqal_cloud import PasqalCloudConnection

from qubosolver import (
    Instance,
    RemoteEmulator,
    Solution,
    analysis,
    drive_shaping,
    embedding,
    solving,
)

# Replace with your username, project id and password on the Pasqal Cloud.
USERNAME = "#TO_PROVIDE"
PROJECT_ID = "#TO_PROVIDE"
PASSWORD = None

Q = np.asarray(
    [
        [-0.2, 0.0, 1.0],
        [0.0, -1.0, 1.5],
        [1.0, 1.5, -0.1],
    ],
    dtype=np.float64,
)
instance = Instance(Q)

if PASSWORD is not None:
    # Setup connection
    connection = PasqalCloudConnection(
        username=USERNAME,
        password=PASSWORD,
        project_id=PROJECT_ID,
    )
else:
    # Use a mock local connection for tutorial
    from qubosolver.utils._local_connection import LocalConnection

    connection = LocalConnection()

# Use a remote emulator backend, on the device chosen above
backend = RemoteEmulator(connection=connection)

register = embedding.blade.embed_for_device(instance, device)
drive = drive_shaping.local_energy_scale.build_drive(instance, register, device=device)
program = solving.analog_quantum_sampling.compile(register, drive, device)
job = backend.run(program)
solution = Solution.from_results(job.results(), instance)

print(analysis.to_dataframe([solution]))

  labels bitstrings  costs  counts  probs
0      0        110   -1.2     911  0.911
1      0        010   -1.0      10  0.010
2      0        100   -0.2      16  0.016
3      0        001   -0.1      60  0.060
4      0        000    0.0       3  0.003


## Solving with a classical approach

We show below an example of solving a QUBO using Tabu search.
More information on classical approaches can be found in the [Classical solvers section of the documentation](https://pasqal-io.github.io/qubo-solver/latest/content/classical/solvers/).

In [ ]:
import numpy as np

from qubosolver import (
    Instance,
    analysis,
    solving,
)

Q = np.asarray(
    [
        [-0.2, 0.0, 1.0],
        [0.0, -1.0, 1.5],
        [1.0, 1.5, -0.1],
    ],
    dtype=np.float64,
)
instance = Instance(Q)

# Solve with tabu search, from 10 random starting bitstrings.
solution = solving.tabu_search.solve(instance, starts=10, time_limit=10.0)

print(analysis.to_dataframe([solution]))

  labels bitstrings  costs  counts  probs
0      0        110   -1.2      10    1.0


Continue to the next tutorial, [Solving a QUBO problem flexibly](02-qubosolver-in-full.ipynb), to save remote jobs and retrieve their results later, and to refine solutions.